In [ ]:
###########################################
# 
# BASE IMAGES- converts downloaded tifs into black and white pngs & tifs
#
########################################## 
import numpy as np
import rasterio
from pathlib import Path

folder = Path("/home/matt/NISAR_cal_eval/code_testing")
out_dir = folder / "base_images"
out_dir.mkdir(exist_ok=True)

tif_paths = sorted(folder.glob("*.tif"))

for path in tif_paths:
    with rasterio.open(path) as src:
        img = src.read(1)
        profile = src.profile

    mag = np.abs(img)
    db = 10 * np.log10(mag + 1e-6)

    valid = db[mag > 0]
    vmin = np.nanpercentile(valid, 2) if len(valid) else db.min()
    vmax = np.nanpercentile(valid, 98) if len(valid) else db.max()

    # Save TIF
    out_tif = out_dir / f"{path.stem}.tif"
    profile.update(dtype=rasterio.float32, count=1)
    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(db.astype(np.float32), 1)

    # Save PNG
    import matplotlib.pyplot as plt
    out_png = out_dir / f"{path.stem}.png"
    plt.imsave(out_png, db, cmap='gray', vmin=vmin, vmax=vmax)

In [ ]:
###########################################
# 
# RATIOS- converts downloaded tifs into nearest neighbors amplitude ratios, pngs and tifs
#
########################################## 
import numpy as np
from pathlib import Path
import re
import rasterio
from rasterio.warp import reproject, Resampling
from datetime import datetime

# CONFIG
gslcPath = Path("/home/matt/NISAR_cal_eval/code_fixing")
workDir  = gslcPath /'ratios/'
workDir.mkdir(exist_ok=True)

# FILENAME PATTERN
pattern = re.compile(
    r".*_(A|D)_[^_]*_[^_]*_[^_]*_[^_]*_"
    r"([0-9]{8}T[0-9]{6})_"
    r"([0-9]{8}T[0-9]{6})_"
    r"([A-Za-z][0-9]{5})_"
    r".*_(HH|HV|VH|VV)"
    r"(?:_freq([AB]))?"
    r"\.tif$"
)

# DISCOVER AND GROUP TIFS
entries = []
for tif in sorted(gslcPath.glob("*.tif")):
    m = pattern.match(tif.name)
    if m:
        look      = m.group(1)
        timestamp = m.group(2)
        pol       = m.group(5)
        freq      = m.group(6)  # 'A', 'B', or None if absent
        date      = datetime.strptime(timestamp[:8], '%Y%m%d')
        entries.append((tif, look, date, pol, freq))

if not entries:
    raise RuntimeError(f"No matching .tif files found in {gslcPath}")

groups = {}
for tif, look, date, pol, freq in entries:
    groups.setdefault(look, {})
    groups[look].setdefault(pol, [])
    groups[look][pol].append((tif, date))

# PROCESS EACH LOOK / POL GROUP
for look_dir, pol_groups in groups.items():
    for pol, items in pol_groups.items():

        items = sorted(items, key=lambda x: x[1])
        nslc  = len(items)

        if nslc < 2:
            print(f"Skipping {look_dir} {pol}: only one image")
            continue

        # Use first file as reference grid
        with rasterio.open(items[0][0]) as ref:
            ny, nx        = ref.height, ref.width
            ref_transform = ref.transform
            ref_crs       = ref.crs

        # Load amplitude stack, aligned to reference grid
        amps = np.zeros((nslc, ny, nx), dtype='float32')

        for i, (tif, date) in enumerate(items):
            with rasterio.open(tif) as src:
                if i == 0:
                    amps[i] = np.abs(src.read(1))
                else:
                    aligned = np.zeros((ny, nx), dtype='float32')
                    reproject(
                        source        = np.abs(src.read(1)),
                        destination   = aligned,
                        src_transform = src.transform,
                        src_crs       = src.crs,
                        dst_transform = ref_transform,
                        dst_crs       = ref_crs,
                        resampling    = Resampling.bilinear
                    )
                    amps[i] = aligned

        # Compute nearest-neighbor amplitude ratios
        with np.errstate(divide='ignore', invalid='ignore'):
            ratios = np.where(amps[1:] > 0, amps[0:-1] / amps[1:], np.nan)

        ni = ratios.shape[0]

        # Plot and save each pair
        for i in range(ni):
            date1 = items[i][1]
            date2 = items[i+1][1]
            t1    = date1.strftime('%Y%m%d')
            t2    = date2.strftime('%Y%m%d')

            if t1 == t2:
                print(f"Skipping same-date pair: {t1} → {t2}")
                continue

            ratio = ratios[i]
            valid = ratio[np.isfinite(ratio) & (ratio > 0)]

            vmin = np.nanpercentile(valid, 2)
            vmax = np.nanpercentile(valid, 98)

            out_tif = workDir / f"ratio_{look_dir}_{pol}_{t1}_to_{t2}.tif"
            out_png = workDir / f"ratio_{look_dir}_{pol}_{t1}_to_{t2}.png"

            with rasterio.open(
                out_tif, "w",
                driver    = "GTiff",
                height    = ratio.shape[0],
                width     = ratio.shape[1],
                count     = 1,
                dtype     = ratio.dtype,
                crs       = ref_crs,
                transform = ref_transform,
            ) as dst:
                dst.write(ratio, 1)

            plt.imsave(out_png, ratio, cmap='gray_r', vmin=vmin, vmax=vmax)

In [ ]:
###########################################
# 
# IGRAMS- converts downloaded tifs into nearest neighbors interferograms, pngs and tifs
#
########################################## 
import numpy as np
from pathlib import Path
import re
import rasterio
from rasterio.warp import reproject, Resampling
from datetime import datetime
import matplotlib.pyplot as plt

# CONFIG
gslcPath = Path("/home/matt/NISAR_cal_eval/code_fixing")
workDir  = gslcPath / 'igrams/'
workDir.mkdir(exist_ok=True)

# FILENAME PATTERN
pattern = re.compile(
    r".*_(A|D)_[^_]*_[^_]*_[^_]*_[^_]*_"
    r"([0-9]{8}T[0-9]{6})_"   # start timestamp
    r"([0-9]{8}T[0-9]{6})_"   # end timestamp
    r"([A-Za-z][0-9]{5})_"   # burst/frame ID
    r".*_(HH|HV|VH|VV)"
    r"(?:_freq([AB]))?"      # optional frequency band suffix
    r"\.tif$"
)

# DISCOVER AND GROUP TIFS
entries = []
for tif in sorted(gslcPath.glob("*.tif")):
    m = pattern.match(tif.name)
    if m:
        look      = m.group(1)
        timestamp = m.group(2)   # start time for date parsing
        pol       = m.group(5)
        freq      = m.group(6)   # 'A', 'B', or None if absent
        date      = datetime.strptime(timestamp[:8], '%Y%m%d')
        entries.append((tif, look, date, pol, freq))

if not entries:
    raise RuntimeError(f"No matching .tif files found in {gslcPath}")

# Group by look direction → polarization
groups = {}
for tif, look, date, pol, freq in entries:
    groups.setdefault(look, {})
    groups[look].setdefault(pol, [])
    groups[look][pol].append((tif, date))

# PROCESS EACH LOOK / POL GROUP
for look_dir, pol_groups in groups.items():
    for pol, items in pol_groups.items():

        items = sorted(items, key=lambda x: x[1])
        nslc  = len(items)

        if nslc < 2:
            print(f"Skipping {look_dir} {pol}: only one image")
            continue

        # Use first file as reference grid
        with rasterio.open(items[0][0]) as ref:
            ny, nx        = ref.height, ref.width
            ref_transform = ref.transform
            ref_crs       = ref.crs

        # Load SLC stack, aligned to reference grid
        slcs = np.zeros((nslc, ny, nx), dtype='complex64')

        for i, (tif, date) in enumerate(items):
            with rasterio.open(tif) as src:
                if i == 0:
                    slcs[i] = src.read(1)
                else:
                    aligned = np.zeros((ny, nx), dtype='complex64')
                    reproject(
                        source        = src.read(1),
                        destination   = aligned,
                        src_transform = src.transform,
                        src_crs       = src.crs,
                        dst_transform = ref_transform,
                        dst_crs       = ref_crs,
                        resampling    = Resampling.bilinear
                    )
                    slcs[i] = aligned

        # Compute interferogram stack
        igram      = slcs[0:-1, :, :] * np.conj(slcs[1:, :, :])
        ni, ny, nx = np.shape(igram)

        # Save each pair as PNG + TIF
        for i in range(ni):
            date1 = items[i][1]
            date2 = items[i+1][1]
            t1    = date1.strftime('%Y%m%d')
            t2    = date2.strftime('%Y%m%d')

            if t1 == t2:
                print(f"Skipping same-date pair: {t1} → {t2}")
                continue

            wrapped = np.angle(igram[i, :, :])

            out_png = workDir / f"wrapped_{look_dir}_{pol}_{t1}_to_{t2}.png"
            out_tif = workDir / f"wrapped_{look_dir}_{pol}_{t1}_to_{t2}.tif"

            plt.imsave(out_png, wrapped, cmap='rainbow', vmin=-np.pi, vmax=np.pi)

            with rasterio.open(
                out_tif, "w",
                driver    = "GTiff",
                height    = wrapped.shape[0],
                width     = wrapped.shape[1],
                count     = 1,
                dtype     = 'float32',
                crs       = ref_crs,
                transform = ref_transform,
            ) as dst:
                dst.write(wrapped.astype('float32'), 1)